In [1]:
import numpy as np
from sklearn.externals.array_api_extra.testing import override

from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 297.76 MB


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import IncrementalPCA

class LogRegModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, n_comp=20):
        self.n_comp = n_comp
        self.inc_pca = None
        self.model = OneVsRestClassifier(LogisticRegression(max_iter=1500))
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression)

    @override
    def full_fit(self, x: np.ndarray, y: np.ndarray):
        if x.shape[0] <= self.n_comp:
            self.inc_pca = IncrementalPCA(n_components=x.shape[0])
            n_batches = 1
        else:
            self.inc_pca = IncrementalPCA(n_components=self.n_comp)
            n_batches = floor(x.shape[0] / self.n_comp)
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))


        for X_batch in np.array_split(x, n_batches):
            self.inc_pca.partial_fit(X_batch)

        x_reduced = self.inc_pca.transform(x)
        self.model.fit(x_reduced, y)

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        x_reduced = self.inc_pca.transform(x)
        return self.model.predict(x_reduced)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(1, x.shape[0] * x.shape[1] * x.shape[2]))
        x_reduced = self.inc_pca.transform(x)
        pred_ = self.model.predict(x_reduced)
        return np.reshape(pred_, shape=(pred_.shape[1]))


In [8]:
# Depth 1 ensemble
ensemble = [LogRegModel(p) for p in planes]
scores = Model.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: None
	Fat Suppression: None
	Training Shape: (64, 12, 512, 512)
	Validation Shape: (43, 12, 512, 512)
	AUC Score: 0.6244025168159355
		ACL: 0.6118421052631579
		MCL: 0.7094594594594594
		Medial Meniscus: 0.6011111111111112
		Lateral Meniscus: 0.6369047619047619
		Medial OA: 0.6196969696969697
		Lateral OA: 0.5575396825396824
		PF OA: 0.6626344086021506
		Effusion: 0.5380434782608696
		Synovitis: 0.5231481481481481
		Baker's: 0.7464285714285714
		Contusion: 0.5333333333333333
		Fracture: 0.7526881720430106

Training Model: 
	Plane: Axial
	Fluid Sensitive: None
	Fat Suppression: None
	Training Shape: (44, 12, 512, 512)
	Validation Shape: (22, 12, 512, 512)
	AUC Score: 0.5668559195648979
		ACL: 0.4572649572649573
		MCL: 0.4473684210526316
		Medial Meniscus: 0.40909090909090906
		Lateral Meniscus: 0.5897435897435898
		Medial OA: 0.5476190476190477
		Lateral OA: 0.8117647058823529
		PF OA: 0.6
		Effusion: 0.4910714285714286
		Synovitis: 0.

In [9]:
print_memory_usage()

Current RAM usage: 5432.50 MB


In [14]:
# Depth 2 ensemble
# fluid sensitive and fat suppression can either be 0 or 1
ensemble = []

for p in planes:
    for i in range(2):
        ensemble.append(LogRegModel(p, i, i, n_comp=20))

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 0
	Fat Suppression: 0
	Training Shape: (39, 12, 512, 512)
	Validation Shape: (18, 12, 512, 512)
	AUC Score: 0.45118804806304796
		ACL: 0.45833333333333326
		MCL: 0.4666666666666667
		Medial Meniscus: 0.3625
		Lateral Meniscus: 0.5
		Medial OA: 0.4230769230769231
		Lateral OA: 0.5
		PF OA: 0.5833333333333334
		Effusion: 0.25
		Synovitis: 0.175
		Baker's: 0.5
		Contusion: 0.48701298701298695
		Fracture: 0.7083333333333334

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 1
	Fat Suppression: 1
	Training Shape: (33, 12, 512, 512)
	Validation Shape: (17, 12, 512, 512)
	AUC Score: 0.497375541125541
		ACL: 0.5069444444444444
		MCL: 0.4642857142857143
		Medial Meniscus: 0.33333333333333337
		Lateral Meniscus: 0.6857142857142857
		Medial OA: 0.4166666666666667
		Lateral OA: 0.4642857142857143
		PF OA: 0.2727272727272727
		Effusion: 0.44696969696969696
		Synovitis: 0.5763888888888888
		Baker's: 0.625
		Contusion: 0.5928571428571427
		Fractur

In [15]:
scores = Model.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Model Score: 0.5753968253968255
	ACL: 0.38333333333333336
	MCL: 0.5
	Medial Meniscus: 0.5714285714285714
	Lateral Meniscus: 0.7666666666666666
	Medial OA: 0.8333333333333333
	Lateral OA: 0.6111111111111112
	PF OA: 0.6785714285714285
	Effusion: 0.49999999999999994
	Synovitis: 0.6666666666666667
	Baker's: 0.3888888888888889
	Contusion: 0.1833333333333333
	Fracture: 0.8214285714285715


In [16]:
scores = Model.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Model Score: 0.5831349206349206
	ACL: 0.5
	MCL: 0.5
	Medial Meniscus: 0.5357142857142857
	Lateral Meniscus: 0.8666666666666667
	Medial OA: 0.6666666666666667
	Lateral OA: 0.5833333333333334
	PF OA: 0.7142857142857142
	Effusion: 0.7142857142857142
	Synovitis: 0.6333333333333333
	Baker's: 0.33333333333333337
	Contusion: 0.2
	Fracture: 0.75


In [17]:
print_memory_usage()

Current RAM usage: 5694.22 MB
